In [9]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn

python_code = """
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib # To save the model

# Load the dataset
try:
    df = pd.read_csv('/content/loan_approval_10000.csv')
except FileNotFoundError:
    print("Error: loan_approval_10000.csv not found. Please ensure it's in the /content/ directory.")
    exit()

# Preprocessing
# Drop Loan_ID as it's not useful for prediction
df = df.drop('Loan_ID', axis=1)

# Handle missing values
# For numerical features, use median
for col in ['Credit_History', 'LoanAmount', 'Loan_Amount_Term']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

# For categorical features, use mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed']:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

# Convert categorical features to numerical using one-hot encoding
# For binary target variable, map 'Y' to 1 and 'N' to 0
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# One-hot encode remaining categorical features
categorical_cols = df.select_dtypes(include='object').columns
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Separate features (X) and target (y)
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost Classifier
model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42)
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Model Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(report)

# Save the trained model
model_filename = 'loan_approval_model.joblib'
joblib.dump(model, model_filename)
print(f"Model saved to {model_filename}")
"""

with open('loan_approval_model.py', 'w') as f:
    f.write(python_code)

!python loan_approval_model.py

Traceback (most recent call last):
  File "/content/loan_approval_model.py", line 19, in <module>
    df = df.drop('Loan_ID', axis=1)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/frame.py", line 5581, in drop
    return super().drop(
           ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/generic.py", line 4788, in drop
    obj = obj._drop_axis(labels, axis, level=level, errors=errors)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/generic.py", line 4830, in _drop_axis
    new_axis = axis.drop(labels, errors=errors)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/core/indexes/base.py", line 7070, in drop
    raise KeyError(f"{labels[mask].tolist()} not found in axis")
KeyError: "['Loan_ID'] not found in axis"


In [10]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║         LOAN APPROVAL PREDICTION - ML PIPELINE (XGBOOST)           ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score
)
from sklearn.ensemble import GradientBoostingClassifier

try:
    from xgboost import XGBClassifier
    USE_XGBOOST = True
    print("  XGBoost library found — using XGBClassifier")
except ImportError:
    USE_XGBOOST = False
    print("   XGBoost not installed — using sklearn GradientBoostingClassifier")

# ─────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────
DATA_PATH   = "/content/loan_approval_10000.csv"
OUTPUT_DIR  = "."
RANDOM_SEED = 42
TEST_SIZE   = 0.20
CV_FOLDS    = 5

np.random.seed(RANDOM_SEED)


# ─────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────
def load_data(path):
    print("\n" + "="*60)
    print("  STEP 1 — LOADING DATA")
    print("="*60)
    df = pd.read_csv(path)
    print(f"  ✔ Rows   : {df.shape[0]:,}")
    print(f"  ✔ Columns: {df.shape[1]}")
    print(f"  Columns  : {list(df.columns)}")
    return df


# ─────────────────────────────────────────────────────────────────────
# 2. EDA
# ─────────────────────────────────────────────────────────────────────
def exploratory_analysis(df):
    print("\n" + "="*60)
    print("  STEP 2 — EXPLORATORY DATA ANALYSIS")
    print("="*60)

    print("\n  [2.1] DATA TYPES")
    print(df.dtypes.to_string())

    print("\n  [2.2] MISSING VALUES")
    mv = df.isnull().sum()
    print(mv[mv > 0] if mv.any() else "  → No missing values ✔")

    print("\n  [2.3] TARGET DISTRIBUTION")
    vc = df["loan_status"].value_counts()
    for label, cnt in vc.items():
        print(f"  {label:12s}: {cnt:5,}  ({cnt/len(df)*100:.1f}%)")

    print("\n  [2.4] DESCRIPTIVE STATISTICS")
    print(df.describe().to_string())

    print("\n  [2.5] CATEGORICAL COLUMNS")
    for col in df.select_dtypes(include="object").columns:
        if col != "loan_status":
            print(f"\n  {col}:\n{df[col].value_counts().to_string()}")


# ─────────────────────────────────────────────────────────────────────
# 3. PREPROCESSING
# ─────────────────────────────────────────────────────────────────────
def preprocess(df):
    print("\n" + "="*60)
    print("  STEP 3 — PREPROCESSING")
    print("="*60)

    df = df.copy()

    # Drop ID
    if "loan_id" in df.columns:
        df.drop(columns=["loan_id"], inplace=True)
        print("  ✔ Dropped 'loan_id'")

    # Encode target
    df["loan_status"] = df["loan_status"].map({"Approved": 1, "Rejected": 0})
    print("  ✔ Target encoded → Approved=1, Rejected=0")

    # Encode categoricals
    le = LabelEncoder()
    for col in df.select_dtypes(include="object").columns:
        df[col] = le.fit_transform(df[col])
        print(f"  ✔ Label-encoded '{col}'")

    # Feature Engineering
    df["total_assets"]      = (df["residential_assets_value"] +
                               df["commercial_assets_value"]  +
                               df["luxury_assets_value"]      +
                               df["bank_asset_value"])
    df["loan_to_income"]    = df["loan_amount"] / (df["income_annum"] + 1)
    df["asset_to_loan"]     = df["total_assets"] / (df["loan_amount"] + 1)
    df["income_per_person"] = df["income_annum"] / (df["no_of_dependents"] + 1)
    print("  ✔ Feature engineering — 4 derived features added")

    X = df.drop(columns=["loan_status"])
    y = df["loan_status"]

    print(f"\n  Final features : {X.shape[1]}")
    print(f"  Final samples  : {X.shape[0]:,}")

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y
    )
    print(f"  Train set      : {X_train.shape[0]:,}")
    print(f"  Test set       : {X_test.shape[0]:,}")

    # Scale
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)
    print("  ✔ StandardScaler applied")

    return X_train_sc, X_test_sc, y_train, y_test, X.columns.tolist(), scaler


# ─────────────────────────────────────────────────────────────────────
# 4. BUILD MODEL
# ─────────────────────────────────────────────────────────────────────
def build_model():
    if USE_XGBOOST:
        return XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
            gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
            use_label_encoder=False, eval_metric="logloss",
            random_state=RANDOM_SEED, n_jobs=-1
        )
    else:
        return GradientBoostingClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, min_samples_split=3, max_features="sqrt",
            random_state=RANDOM_SEED
        )


# ─────────────────────────────────────────────────────────────────────
# 5. TRAIN + CROSS-VALIDATE
# ─────────────────────────────────────────────────────────────────────
def train_model(model, X_train, y_train):
    print("\n" + "="*60)
    print("  STEP 4 — TRAINING & CROSS-VALIDATION")
    print("="*60)

    cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    print(f"\n  Running {CV_FOLDS}-Fold Stratified CV ...")

    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring="accuracy",  n_jobs=-1)
    cv_roc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc",   n_jobs=-1)
    cv_f1  = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1",        n_jobs=-1)

    print(f"\n  CV Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
    print(f"  CV ROC-AUC  : {cv_roc.mean():.4f} ± {cv_roc.std():.4f}")
    print(f"  CV F1-Score : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

    print("\n  Training final model ...")
    model.fit(X_train, y_train)
    print("  ✔ Training complete")

    return model, {"cv_accuracy": cv_acc, "cv_roc_auc": cv_roc, "cv_f1": cv_f1}


# ─────────────────────────────────────────────────────────────────────
# 6. EVALUATE
# ─────────────────────────────────────────────────────────────────────
def evaluate_model(model, X_test, y_test):
    print("\n" + "="*60)
    print("  STEP 5 — EVALUATION ON TEST SET")
    print("="*60)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc      = accuracy_score(y_test, y_pred)
    roc_auc  = roc_auc_score(y_test, y_prob)
    f1       = f1_score(y_test, y_pred)
    avg_prec = average_precision_score(y_test, y_prob)

    print(f"\n  Accuracy      : {acc:.4f}  ({acc*100:.2f}%)")
    print(f"  ROC-AUC       : {roc_auc:.4f}")
    print(f"  F1-Score      : {f1:.4f}")
    print(f"  Avg Precision : {avg_prec:.4f}")

    print("\n  CLASSIFICATION REPORT:")
    print(classification_report(y_test, y_pred,
                                target_names=["Rejected (0)", "Approved (1)"]))

    cm = confusion_matrix(y_test, y_pred)
    print("  CONFUSION MATRIX:")
    print(f"  ┌────────────────────────────┐")
    print(f"  │              Predicted     │")
    print(f"  │           Rej   App        │")
    print(f"  │ Actual Rej {cm[0,0]:4d}  {cm[0,1]:4d}      │")
    print(f"  │        App {cm[1,0]:4d}  {cm[1,1]:4d}      │")
    print(f"  └────────────────────────────┘")

    return {"y_pred": y_pred, "y_prob": y_prob, "accuracy": acc,
            "roc_auc": roc_auc, "f1": f1, "avg_prec": avg_prec, "cm": cm}


# ─────────────────────────────────────────────────────────────────────
# 7. DASHBOARD
# ─────────────────────────────────────────────────────────────────────
def plot_dashboard(model, X_test, y_test, metrics, cv_metrics, feature_names, df_orig):
    print("\n" + "="*60)
    print("  STEP 6 — GENERATING DASHBOARD")
    print("="*60)

    sns.set_theme(style="whitegrid")
    PALETTE     = {"Approved": "#2ecc71", "Rejected": "#e74c3c"}
    TITLE_COLOR = "#f0f0f0"
    AX_BG       = "#1a1a2e"
    GRID_COLOR  = "#2a2a3e"

    fig = plt.figure(figsize=(24, 26))
    fig.patch.set_facecolor("#0f1117")
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

    def style_ax(ax, title=""):
        ax.set_facecolor(AX_BG)
        ax.tick_params(colors=TITLE_COLOR, labelsize=9)
        ax.xaxis.label.set_color(TITLE_COLOR)
        ax.yaxis.label.set_color(TITLE_COLOR)
        ax.spines[:].set_color(GRID_COLOR)
        if title:
            ax.set_title(title, color=TITLE_COLOR, fontsize=11, fontweight="bold", pad=10)

    # Header
    ax_hdr = fig.add_subplot(gs[0, :])
    ax_hdr.set_facecolor("#16213e"); ax_hdr.axis("off")
    ax_hdr.text(0.5, 0.5,
        f"🏦  LOAN APPROVAL PREDICTION — XGBoost ML Pipeline\n"
        f"Dataset: 10,000 samples  |  Features: {len(feature_names)}  |  "
        f"Test Accuracy: {metrics['accuracy']*100:.2f}%  |  "
        f"ROC-AUC: {metrics['roc_auc']:.4f}  |  F1: {metrics['f1']:.4f}",
        transform=ax_hdr.transAxes, fontsize=14, color=TITLE_COLOR,
        ha="center", va="center", fontweight="bold")

    # 1. Target Distribution
    ax1  = fig.add_subplot(gs[1, 0])
    vc   = df_orig["loan_status"].value_counts()
    bars = ax1.bar(vc.index, vc.values,
                   color=[PALETTE.get(l, "#3498db") for l in vc.index],
                   edgecolor="white", linewidth=0.6)
    for bar, val in zip(bars, vc.values):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f"{val:,}\n({val/len(df_orig)*100:.1f}%)",
                 ha="center", va="bottom", color=TITLE_COLOR, fontsize=9)
    style_ax(ax1, "1 — Target Distribution")
    ax1.set_xlabel("Loan Status"); ax1.set_ylabel("Count")

    # 2. Confusion Matrix
    ax2 = fig.add_subplot(gs[1, 1])
    sns.heatmap(metrics["cm"], annot=True, fmt="d", cmap="Blues", ax=ax2,
                xticklabels=["Rejected", "Approved"],
                yticklabels=["Rejected", "Approved"],
                cbar_kws={"shrink": 0.8})
    ax2.set_xlabel("Predicted", color=TITLE_COLOR)
    ax2.set_ylabel("Actual", color=TITLE_COLOR)
    style_ax(ax2, "2 — Confusion Matrix")

    # 3. ROC Curve
    ax3 = fig.add_subplot(gs[1, 2])
    fpr, tpr, _ = roc_curve(y_test, metrics["y_prob"])
    ax3.plot(fpr, tpr, color="#3498db", lw=2, label=f"AUC = {metrics['roc_auc']:.4f}")
    ax3.plot([0,1],[0,1], color="#7f8c8d", lw=1.2, linestyle="--", label="Random")
    ax3.fill_between(fpr, tpr, alpha=0.15, color="#3498db")
    ax3.set_xlabel("False Positive Rate"); ax3.set_ylabel("True Positive Rate")
    ax3.legend(loc="lower right", facecolor=AX_BG, labelcolor=TITLE_COLOR, fontsize=9)
    style_ax(ax3, "3 — ROC Curve")

    # 4. Precision-Recall
    ax4 = fig.add_subplot(gs[2, 0])
    prec, rec, _ = precision_recall_curve(y_test, metrics["y_prob"])
    ax4.plot(rec, prec, color="#e74c3c", lw=2, label=f"AP = {metrics['avg_prec']:.4f}")
    ax4.fill_between(rec, prec, alpha=0.15, color="#e74c3c")
    ax4.set_xlabel("Recall"); ax4.set_ylabel("Precision")
    ax4.legend(facecolor=AX_BG, labelcolor=TITLE_COLOR, fontsize=9)
    style_ax(ax4, "4 — Precision-Recall Curve")

    # 5. Cross-Validation
    ax5 = fig.add_subplot(gs[2, 1])
    x   = np.arange(CV_FOLDS); w = 0.28
    ax5.bar(x - w, cv_metrics["cv_accuracy"], width=w, label="Accuracy",
            color="#3498db", alpha=0.85, edgecolor="white")
    ax5.bar(x,     cv_metrics["cv_roc_auc"],  width=w, label="ROC-AUC",
            color="#2ecc71", alpha=0.85, edgecolor="white")
    ax5.bar(x + w, cv_metrics["cv_f1"],       width=w, label="F1-Score",
            color="#e74c3c", alpha=0.85, edgecolor="white")
    ax5.set_xticks(x)
    ax5.set_xticklabels([f"Fold {i+1}" for i in range(CV_FOLDS)],
                        color=TITLE_COLOR, fontsize=8)
    ax5.set_ylim(0.8, 1.02)
    ax5.legend(facecolor=AX_BG, labelcolor=TITLE_COLOR, fontsize=9)
    style_ax(ax5, f"5 — {CV_FOLDS}-Fold Cross-Validation")

    # 6. Feature Importance
    ax6 = fig.add_subplot(gs[2, 2])
    if hasattr(model, "feature_importances_"):
        fi = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=True)
        colors_fi = ["#e74c3c" if v >= fi.quantile(0.75) else "#3498db" for v in fi.values]
        fi.plot(kind="barh", ax=ax6, color=colors_fi, edgecolor="white", linewidth=0.5)
        ax6.set_xlabel("Importance Score")
    style_ax(ax6, "6 — Feature Importance")

    # 7. Score Distribution
    ax7   = fig.add_subplot(gs[3, 0])
    prob  = metrics["y_prob"]; y_arr = np.array(y_test)
    ax7.hist(prob[y_arr == 1], bins=40, alpha=0.7, color="#2ecc71", label="Approved")
    ax7.hist(prob[y_arr == 0], bins=40, alpha=0.7, color="#e74c3c", label="Rejected")
    ax7.axvline(0.5, color="white", lw=1.5, linestyle="--", label="Threshold 0.5")
    ax7.set_xlabel("Predicted Probability"); ax7.set_ylabel("Count")
    ax7.legend(facecolor=AX_BG, labelcolor=TITLE_COLOR, fontsize=9)
    style_ax(ax7, "7 — Prediction Score Distribution")

    # 8. CIBIL Score Distribution
    ax8 = fig.add_subplot(gs[3, 1])
    for status, color in [("Approved","#2ecc71"), ("Rejected","#e74c3c")]:
        ax8.hist(df_orig[df_orig["loan_status"]==status]["cibil_score"],
                 bins=30, alpha=0.7, color=color, label=status, edgecolor="white")
    ax8.set_xlabel("CIBIL Score"); ax8.set_ylabel("Count")
    ax8.legend(facecolor=AX_BG, labelcolor=TITLE_COLOR, fontsize=9)
    style_ax(ax8, "8 — CIBIL Score by Loan Status")

    # 9. Scorecard
    ax9 = fig.add_subplot(gs[3, 2])
    ax9.set_facecolor(AX_BG); ax9.axis("off")
    rows = [
        ("Metric",        "CV (mean)",                              "Test"),
        ("─"*14,          "─"*14,                                   "─"*12),
        ("Accuracy",      f"{cv_metrics['cv_accuracy'].mean():.4f}", f"{metrics['accuracy']:.4f}"),
        ("ROC-AUC",       f"{cv_metrics['cv_roc_auc'].mean():.4f}",  f"{metrics['roc_auc']:.4f}"),
        ("F1-Score",      f"{cv_metrics['cv_f1'].mean():.4f}",       f"{metrics['f1']:.4f}"),
        ("Avg Precision", "—",                                        f"{metrics['avg_prec']:.4f}"),
        ("",              "",                                         ""),
        ("Train samples", "8,000",                                    "—"),
        ("Test samples",  "—",                                        "2,000"),
    ]
    for ri, row in enumerate(rows):
        for ci, val in enumerate(row):
            ax9.text(ci*0.33 + 0.02, 1 - ri*0.1, val,
                     transform=ax9.transAxes, fontsize=9,
                     color="#f39c12" if ri == 0 else TITLE_COLOR,
                     va="top", fontweight="bold" if ri == 0 else "normal",
                     fontfamily="monospace")
    style_ax(ax9, "9 — Model Scorecard")

    out = f"{OUTPUT_DIR}/loan_approval_dashboard.png"
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()
    print(f"  ✔ Dashboard saved → {out}")


# ─────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────
def main():
    print("\n╔" + "═"*58 + "╗")
    print("║     LOAN APPROVAL PREDICTION — XGBoost ML PIPELINE      ║")
    print("╚" + "═"*58 + "╝")

    df_orig = load_data(DATA_PATH)
    exploratory_analysis(df_orig)
    X_train, X_test, y_train, y_test, feature_names, scaler = preprocess(df_orig)

    model = build_model()
    algo  = "XGBClassifier" if USE_XGBOOST else "GradientBoostingClassifier"
    print(f"\n  Model  : {algo}")
    print(f"  Params : n_estimators=300, max_depth=6, lr=0.05, subsample=0.8")

    model, cv_metrics = train_model(model, X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    plot_dashboard(model, X_test, y_test, metrics, cv_metrics, feature_names, df_orig)

    print("\n" + "="*60)
    print("   FINAL RESULTS")
    print("="*60)
    print(f"  Algorithm     : {algo}")
    print(f"  Features      : {len(feature_names)} (incl. 4 engineered)")
    print(f"  CV Accuracy   : {cv_metrics['cv_accuracy'].mean()*100:.2f}% ± {cv_metrics['cv_accuracy'].std()*100:.2f}%")
    print(f"  CV ROC-AUC    : {cv_metrics['cv_roc_auc'].mean():.4f} ± {cv_metrics['cv_roc_auc'].std():.4f}")
    print(f"  Test Accuracy : {metrics['accuracy']*100:.2f}%")
    print(f"  Test ROC-AUC  : {metrics['roc_auc']:.4f}")
    print(f"  Test F1-Score : {metrics['f1']:.4f}")
    print("="*60 + "\n")


if __name__ == "__main__":
    main()

  XGBoost library found — using XGBClassifier

╔══════════════════════════════════════════════════════════╗
║     LOAN APPROVAL PREDICTION — XGBoost ML PIPELINE      ║
╚══════════════════════════════════════════════════════════╝

  STEP 1 — LOADING DATA
  ✔ Rows   : 10,000
  ✔ Columns: 13
  Columns  : ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']

  STEP 2 — EXPLORATORY DATA ANALYSIS

  [2.1] DATA TYPES
loan_id                      int64
no_of_dependents             int64
education                   object
self_employed               object
income_annum                 int64
loan_amount                  int64
loan_term                    int64
cibil_score                  int64
residential_assets_value     int64
commercial_assets_value      int64
luxury_assets_value          int64
bank_asset_value    